# PandasDataFrameOutputParser - 자연어로 DataFrame을 조회해요

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요.

- `PandasDataFrameOutputParser`로 Pandas DataFrame 스키마를 LLM에 전달할 수 있어요
- 자연어 질문으로 컬럼 조회, 행 필터링, 집계 연산을 수행할 수 있어요
- 비개발자도 데이터에 접근할 수 있는 자연어 인터페이스를 구현할 수 있어요

## 사전 지식

- **Pandas DataFrame**: 행(row)과 열(column)로 구성된 표 형태의 데이터 구조예요. 엑셀 시트와 유사해요
- **PandasDataFrameOutputParser**: DataFrame의 컬럼명과 타입 정보를 LLM에 제공하고, LLM이 적절한 Pandas 연산을 선택하도록 유도하는 파서예요

## 파이프라인 흐름

```mermaid
flowchart LR
    DF["기존 DataFrame<br/>(컬럼·타입 정보)"] -->|"스키마 제공"| P["PandasDataFrame<br/>OutputParser"]
    P -->|"형식 지침 생성"| PT["PromptTemplate"]
    PT -->|"자연어 질문 포함"| LLM["ChatOpenAI"]
    LLM -->|"Pandas 연산 지정"| P
    P -->|"실제 연산 실행"| R["조회 결과<br/>(Series·dict·float)"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class DF storage
    class P,PT,LLM process
    class R output
```

## 지원하는 조회 유형

| 조회 유형 | 예시 질문 | 반환 형태 |
|-----------|-----------|-----------|
| 컬럼 조회 | "가격 컬럼을 조회해주세요" | `pandas.Series` |
| 특정 행 조회 | "첫 번째 행을 조회해주세요" | `pandas.Series` |
| 집계 연산 | "평균 가격을 계산해주세요" | `dict` |

> **주의**: 이 파서는 DataFrame 데이터를 LLM에 직접 보내지 않아요. 컬럼명과 타입 정보(스키마)만 전달하고, LLM이 어떤 Pandas 연산을 쓸지 결정해요. 실제 계산은 파서가 로컬에서 수행해요.

In [1]:
from dotenv import load_dotenv

load_dotenv()


True

## 실용 예제: 전자제품 판매 데이터 분석

전자제품 판매 데이터를 자연어로 조회하는 시나리오입니다.


In [2]:
import pandas as pd
from langchain.output_parsers import PandasDataFrameOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 1단계: 샘플 DataFrame 생성
data = {
    "제품명": ["노트북", "스마트폰", "태블릿", "스마트워치", "이어폰"],
    "카테고리": ["컴퓨터", "모바일", "모바일", "웨어러블", "액세서리"],
    "가격": [1200000, 950000, 650000, 350000, 180000],
    "판매량": [45, 120, 78, 95, 210],
    "재고": [15, 8, 25, 40, 150]
}

df = pd.DataFrame(data)

print("=" * 60)
print("📊 전자제품 판매 데이터")
print("=" * 60)
print(df)
print()

📊 전자제품 판매 데이터
     제품명  카테고리       가격  판매량   재고
0    노트북   컴퓨터  1200000   45   15
1   스마트폰   모바일   950000  120    8
2    태블릿   모바일   650000   78   25
3  스마트워치  웨어러블   350000   95   40
4    이어폰  액세서리   180000  210  150



In [3]:
# ---------------------------------------------------
# PandasDataFrameOutputParser 설정 및 체인 구성
# ---------------------------------------------------

# 1단계: PandasDataFrameOutputParser 생성
# dataframe=df: DataFrame의 컬럼명·타입 정보를 파서에 등록
# 🔑 핵심 개념: 실제 데이터(값)는 LLM에 전달되지 않아요 → 개인정보 보호에 안전
# 컬럼 스키마(컬럼명 + 타입)만 형식 지침으로 제공되고 실제 계산은 파서가 로컬에서 수행
parser = PandasDataFrameOutputParser(dataframe=df)

# 2단계: 프롬프트 템플릿
# format_instructions에 DataFrame 컬럼 정보와 가능한 연산이 자동으로 포함됨
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\nIMPORTANT: Do not wrap your output in quotes.\n{question}\n",
    input_variables=["question"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# 3단계: 체인 구성
# gpt-4o-mini: Pandas 연산 선택에 충분한 성능
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = prompt | model | parser

---

## 자연어 쿼리 예제

이제 실제로 자연어 질문을 던져볼게요. LLM이 질문을 분석해서 적절한 Pandas 연산을 선택하고, 파서가 그 연산을 실행해요.

> 🎯 **강의 포인트**: LLM은 "어떤 연산을 쓸지"만 결정하고, 실제 연산은 파서가 로컬에서 실행해요. LLM이 연산 이름을 잘못 선택하면 파서 오류가 날 수 있으니, 질문을 구체적으로 작성하는 것이 좋아요.

> 💡 **실무 팁**: 질문은 한국어로 해도 괜찮아요. 단, 컬럼명이 한국어인 경우 프롬프트에서 컬럼명을 명시적으로 언급하면 더 정확하게 인식해요.

In [4]:
df.columns

Index(['제품명', '카테고리', '가격', '판매량', '재고'], dtype='object')

In [5]:
# ---------------------------------------------------
# 자연어 쿼리 1: 특정 컬럼 전체 조회
# ---------------------------------------------------

print("=" * 60)
print("🔍 쿼리 1: 가격 컬럼 조회")
print("=" * 60)
# 반환 형태: dict → {컬럼명: pandas.Series}
result1 = chain.invoke({"question": "가격 컬럼을 조회해 주세요."})
print(result1)
print()

🔍 쿼리 1: 가격 컬럼 조회
{'가격': 0    1200000
1     950000
2     650000
3     350000
4     180000
Name: 가격, dtype: int64}



In [6]:
# ---------------------------------------------------
# 자연어 쿼리 2: 특정 행 조회
# ---------------------------------------------------

print("=" * 60)
print("🔍 쿼리 2: 첫 번째 제품 정보")
print("=" * 60)
# 반환 형태: dict → {행 인덱스: pandas.Series}
result2 = chain.invoke({"question": "첫 번째 제품의 정보를 조회해주세요."})
print(result2)
print()

🔍 쿼리 2: 첫 번째 제품 정보
{'0': 제품명         노트북
카테고리        컴퓨터
가격      1200000
판매량          45
재고           15
Name: 0, dtype: object}



In [7]:
# ---------------------------------------------------
# 자연어 쿼리 3: 집계 함수 실행
# ---------------------------------------------------

print("=" * 60)
print("🔍 쿼리 3: 평균 가격 계산")
print("=" * 60)
# 반환 형태: dict → {"mean": 집계값} 또는 {"sum": 값} 등
result3 = chain.invoke({"question": "모든 제품의 평균 가격을 계산해주세요."})
print(f"평균 가격: {result3}")
print()

🔍 쿼리 3: 평균 가격 계산
평균 가격: {'mean': 666000.0}



## 핵심 요약

| 항목 | 설명 |
|------|------|
| **역할** | 자연어 질문 → Pandas 연산 선택 → 결과 반환 |
| **LLM에 전달** | 컬럼명과 타입 정보(스키마)만, 실제 데이터는 전달 안 함 |
| **반환 형태** | 컬럼 조회: `Series`, 집계: `dict`, 행 조회: `Series` |
| **활용** | 비개발자용 데이터 조회 인터페이스, 데이터 분석 자동화 |

> ⚠️ **자주 하는 실수**: `PandasDataFrameOutputParser`는 DataFrame을 교체할 때 파서도 새로 생성해야 해요. 기존 파서에 새 DataFrame을 할당해도 반영되지 않아요.

> 💡 **실무 팁**: 복잡한 필터링(예: "가격이 100만원 이상인 제품")은 LLM이 잘못된 연산을 선택할 수 있어요. 이런 경우에는 LangChain의 `PandasAI`나 SQL 기반 접근을 고려하세요.

### 활용 시나리오

```mermaid
flowchart TD
    U["사용자 자연어 질문"] --> C{"질문 유형"}
    C -->|"컬럼 조회"| A["df.컬럼명<br/>전체 컬럼 값"]
    C -->|"특정 행 조회"| B["df.iloc(n)<br/>행 데이터"]
    C -->|"집계 연산"| D["df.컬럼.mean()<br/>또는 .sum() 등"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class U input
    class C process
    class A,B,D output
```

## 다음 노트북 예고

**노트북 06 - OutputFixingParser** 에서는 파싱이 실패했을 때 LLM이 자동으로 오류를 수정해주는 방법을 배워요. 프로덕션 환경에서 안정성을 높이는 핵심 기법이에요.

### 연습 1: 학생 성적 데이터 조회

새로운 DataFrame을 생성하고 `PandasDataFrameOutputParser`를 사용하여 자연어로 데이터를 조회해보세요.

**요구사항:**
- 아래 데이터로 DataFrame을 생성하세요:

| 이름 | 과목 | 점수 | 학년 |
|------|------|------|------|
| 김철수 | 수학 | 95 | 1 |
| 이영희 | 영어 | 88 | 2 |
| 박민수 | 수학 | 72 | 1 |
| 정수진 | 과학 | 91 | 3 |
| 최지원 | 영어 | 85 | 2 |

- `PandasDataFrameOutputParser`를 사용하여 다음 3가지 자연어 질문을 실행하세요:
  1. "점수 컬럼을 조회해주세요."
  2. "첫 번째 학생의 정보를 조회해주세요."
  3. "점수의 평균을 계산해주세요."

In [8]:
# ============================================================
# TODO: 학생 성적 데이터 조회하기
# 힌트: PandasDataFrameOutputParser(dataframe=student_df)로 파서를 생성하고
#       자연어로 컬럼 조회 / 행 조회 / 집계 쿼리를 각각 실행하세요
# 예상 결과: 각 쿼리에 맞는 Series 또는 dict가 반환됨
# ============================================================

# 연습 1 풀이

import pandas as pd
from langchain.output_parsers import PandasDataFrameOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 1단계: 학생 성적 DataFrame 생성
student_data = {
    "이름": ["김철수", "이영희", "박민수", "정수진", "최지원"],
    "과목": ["수학", "영어", "수학", "과학", "영어"],
    "점수": [95, 88, 72, 91, 85],
    "등급": [1, 2, 1, 3, 2],
}
student_df = pd.DataFrame(student_data)

print("=" * 60)
print("학생 성적 데이터")
print("=" * 60)
print(student_df)
print()

# 2단계: PandasDataFrameOutputParser 생성
# DataFrame의 컬럼 정보(이름, 과목, 점수, 학년)가 형식 지침에 포함됨
parser = PandasDataFrameOutputParser(dataframe=student_df)

# 3단계: 프롬프트 및 체인 구성
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\nIMPORTANT: Do not wrap your output in quotes.\n{question}\n",
    input_variables=["question"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = prompt | model | parser

# 4단계: 3가지 자연어 쿼리 실행
questions = [
    "점수 컬럼을 조회해주세요.",
    "첫 번째 학생의 정보를 조회해주세요.",
    "점수의 평균을 계산해주세요.",
]

for i, q in enumerate(questions, 1):
    print(f"[쿼리 {i}] {q}")
    result = chain.invoke({"question": q})
    print(f"결과: {result}")
    print()

학생 성적 데이터
    이름  과목  점수  등급
0  김철수  수학  95   1
1  이영희  영어  88   2
2  박민수  수학  72   1
3  정수진  과학  91   3
4  최지원  영어  85   2

[쿼리 1] 점수 컬럼을 조회해주세요.
결과: {'점수': 0    95
1    88
2    72
3    91
4    85
Name: 점수, dtype: int64}

[쿼리 2] 첫 번째 학생의 정보를 조회해주세요.
결과: {'0': 이름    김철수
과목     수학
점수     95
등급      1
Name: 0, dtype: object}

[쿼리 3] 점수의 평균을 계산해주세요.
결과: {'mean': 86.2}



### 연습 2: 커피숍 메뉴 데이터 분석

커피숍 메뉴 데이터를 생성하고 다양한 자연어 쿼리를 실행해보세요.

**요구사항:**
- 아래 데이터로 DataFrame을 생성하세요:

| 메뉴명 | 카테고리 | 가격 | 칼로리 | 인기도 |
|--------|---------|------|--------|--------|
| 아메리카노 | 커피 | 4500 | 10 | 95 |
| 카페라떼 | 커피 | 5000 | 180 | 88 |
| 녹차라떼 | 차 | 5500 | 200 | 72 |
| 초코프라페 | 블렌디드 | 6000 | 350 | 65 |
| 자몽에이드 | 에이드 | 5500 | 120 | 80 |

- 다음 2가지 자연어 질문을 실행하세요:
  1. "가격 컬럼을 조회해주세요."
  2. "칼로리의 평균을 계산해주세요."

In [9]:
# ============================================================
# TODO: 커피숍 메뉴 데이터 분석하기
# 힌트: 새 DataFrame을 생성하고 PandasDataFrameOutputParser를 연결하세요
# 예상 결과: 가격 컬럼 조회와 칼로리 평균이 각각 출력됨
# ============================================================

# 연습 2 풀이

import pandas as pd
from langchain.output_parsers import PandasDataFrameOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 1단계: 커피숍 메뉴 DataFrame 생성
coffee_data = {
    "메뉴명": ["아메리카노", "카페라떼", "녹차라떼", "초코프라페", "자몽에이드"],
    "카테고리": ["커피", "커피", "차", "블렌디드", "에이드"],
    "가격": [4500, 5000, 5500, 6000, 5500],
    "칼로리": [10, 180, 200, 350, 120],
    "인기도": [95, 88, 72, 65, 80],
}
coffee_df = pd.DataFrame(coffee_data)

print("=" * 60)
print("커피숍 메뉴 데이터")
print("=" * 60)
print(coffee_df)
print()

# 2단계: 새 DataFrame으로 파서를 별도 생성
# 파서는 연결된 DataFrame에만 접근 가능 → 다른 DataFrame은 새 파서 생성 필요
coffee_parser = PandasDataFrameOutputParser(dataframe=coffee_df)

# 3단계: 프롬프트 및 체인 구성
coffee_prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\nIMPORTANT: Do not wrap your output in quotes.\n{question}\n",
    input_variables=["question"],
    partial_variables={"format_instructions": coffee_parser.get_format_instructions()},
)

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
coffee_chain = coffee_prompt | model | coffee_parser

# 4단계: 자연어 쿼리 실행
queries = [
    "가격 컬럼을 조회해주세요.",
    "칼로리의 평균을 계산해주세요.",
]

for i, q in enumerate(queries, 1):
    print(f"[쿼리 {i}] {q}")
    result = coffee_chain.invoke({"question": q})
    print(f"결과: {result}")
    print()

커피숍 메뉴 데이터
     메뉴명  카테고리    가격  칼로리  인기도
0  아메리카노    커피  4500   10   95
1   카페라떼    커피  5000  180   88
2   녹차라떼     차  5500  200   72
3  초코프라페  블렌디드  6000  350   65
4  자몽에이드   에이드  5500  120   80

[쿼리 1] 가격 컬럼을 조회해주세요.
결과: {'가격': 0    4500
1    5000
2    5500
3    6000
4    5500
Name: 가격, dtype: int64}

[쿼리 2] 칼로리의 평균을 계산해주세요.
결과: {'mean': 172.0}

